In [ ]:
import gradio as gr
import requests
import dotenv
import os
import random

dotenv.load_dotenv()
OPEN_AI_KEY2 = os.getenv('OPEN_AI_KEY2')
AZURE_SPEECH_KEY = os.getenv('AZURE_SPEECH_KEY')

OPENAI_ENDPOINT = "https://fimtrus-foundry.cognitiveservices.azure.com/openai/deployments/gpt-4.1/chat/completions?api-version=2025-01-01-preview"
OPENAI_MODEL    = "9ai043-gpt-4o-mini"


# ============================================================
# 파트 정의
# ============================================================
PART_CHOICES = {
    "TOEIC Speaking": ["파트 1 (사진 묘사)", "파트 2 (질문에 답하기)", "파트 3 (정보 제공)", "파트 4 (의견 제시)"],
    "OPIc":           ["자기소개", "일상 묘사", "경험 말하기", "롤플레이"],
}

# 파트별 제한 시간(초)
TIME_LIMIT = {
    "파트 1 (사진 묘사)":     45,
    "파트 2 (질문에 답하기)": 30,
    "파트 3 (정보 제공)":     30,
    "파트 4 (의견 제시)":     60,
    "자기소개":    60,
    "일상 묘사":   60,
    "경험 말하기": 90,
    "롤플레이":    90,
}

# 파트별 예상 문제 풀 (각 3개, 실행 시 랜덤 1개 선택)
SAMPLE_QUESTIONS = {
    "파트 1 (사진 묘사)": [
        "Describe the picture in as much detail as possible.",
        "Look at the picture and describe what you see.",
        "Tell me everything you notice in this photograph.",
    ],
    "파트 2 (질문에 답하기)": [
        "What do you usually do on weekends?",
        "How do you commute to work or school?",
        "What kind of food do you enjoy eating most?",
    ],
    "파트 3 (정보 제공)": [
        "Imagine you work at a travel agency. A customer calls asking about tours to Japan. Provide details about available packages.",
        "You are a customer service representative. A caller asks about your store's return policy. Explain it clearly.",
        "You work at a hotel front desk. A guest asks about facilities and check-out time. Answer their questions.",
    ],
    "파트 4 (의견 제시)": [
        "Do you think working from home is more productive than working in an office? Why or why not?",
        "Some people prefer living in a big city; others prefer the countryside. Which do you prefer and why?",
        "Is it important for children to learn a second language at an early age? Give your opinion.",
    ],
    "자기소개": [
        "Tell me about yourself, including your job, hobbies, and daily routine.",
        "Introduce yourself and describe where you live.",
        "Talk about your family and what you like to do together.",
    ],
    "일상 묘사": [
        "Describe what a typical weekday looks like for you.",
        "Talk about your morning routine in detail.",
        "Describe your neighborhood and what you like about it.",
    ],
    "경험 말하기": [
        "Tell me about a memorable trip you have taken.",
        "Describe a challenging situation you faced at work or school and how you handled it.",
        "Talk about a time when you learned something new and important.",
    ],
    "롤플레이": [
        "You want to book a table at a restaurant for 4 people this Saturday evening. Call the restaurant and make a reservation.",
        "You bought a jacket online but received the wrong size. Call customer service to resolve the issue.",
        "You are planning a surprise birthday party for a friend. Call a bakery to order a custom cake.",
    ],
}

# 시험 유형별 평가 기준
EVAL_CRITERIA = {
    "TOEIC Speaking": """
- 발음 및 억양 (Pronunciation & Intonation): 명확한 발음, 자연스러운 강세와 억양
- 내용의 완성도 (Content): 문제에 대한 적절하고 논리적인 답변
- 어휘 (Vocabulary): 다양하고 적절한 어휘 사용
- 문법 (Grammar): 시제, 관사, 전치사 등 정확한 문법
- 유창성 (Fluency): 자연스러운 속도, 불필요한 멈춤 최소화
- 등급 기준: 0~8등급""",
    "OPIc": """
- 과제 완성도 (Task Completion): 질문에 충실히 답하였는가
- 내용의 구체성 (Content Detail): 구체적인 예시, 경험, 세부사항 포함
- 어휘 (Vocabulary): 주제에 맞는 다양한 어휘
- 문법 (Grammar): 복잡한 문장 구조 사용 능력
- 유창성 (Fluency): 자연스러운 말의 흐름
- 등급 기준: NL/NM/NH/IL/IM1/IM2/IM3/IH/AL""",
}


# ============================================================
# STT: 음성 파일 → 영어 텍스트 (Azure Speech)
# ============================================================
def request_stt(audio_path):
    endpoint = (
        "https://eastus.stt.speech.microsoft.com"
        "/speech/recognition/conversation/cognitiveservices/v1"
        "?language=en-US&format=detailed"
    )
    headers = {
        "Ocp-Apim-Subscription-Key": AZURE_SPEECH_KEY
        "Content-Type": "audio/wav",
    }
    with open(audio_path, "rb") as f:
        audio_data = f.read()

    response = requests.post(endpoint, headers=headers, data=audio_data)
    if not response.ok:
        print(f"[STT Error] {response.status_code} - {response.text}")
        return None

    data = response.json()
    if "NBest" not in data or not data["NBest"]:
        return None
    return data["NBest"][0]["Display"]


# ============================================================
# OpenAI 공통 호출
# message_list를 받아 assistant의 응답 문자열을 반환
# ============================================================
def call_openai(message_list, max_tokens=2000):
    headers = {
        "Authorization": OPEN_AI_KEY2,
        "Content-Type": "application/json",
    }
    body = {
        "messages": message_list,
        "max_tokens": max_tokens,
        "temperature": 0.7,
        "top_p": 0.95,
        "model": OPENAI_MODEL,
    }
    response = requests.post(OPENAI_ENDPOINT, headers=headers, json=body)
    data = response.json()
    return data["choices"][0]["message"]["content"]


# ============================================================
# 문제 생성: 파트에서 랜덤 1개 추출
# ============================================================
def generate_question(part):
    question = random.choice(SAMPLE_QUESTIONS[part])
    time_sec  = TIME_LIMIT[part]
    time_str  = f"{time_sec // 60:02d}:{time_sec % 60:02d}"
    return question, time_str


# ============================================================
# GPT 평가 요청
# 현재 문제 + 사용자 발화 + 히스토리를 함께 전달
# ============================================================
def request_openai_evaluation(transcript, exam_type, current_question, chat_history):
    system_prompt = f"""당신은 {exam_type} 영어 말하기 시험 전문 평가관입니다.
사용자가 아래 문제에 대해 말한 영어 답변을 {exam_type} 기준으로 상세하게 평가해주세요.

## 출제된 문제
{current_question}

## 평가 기준
{EVAL_CRITERIA[exam_type]}

## 평가 형식 (반드시 한국어로 아래 형식을 따르세요)

### 📝 인식된 발화 내용
(사용자가 말한 텍스트 그대로)

### 🎯 종합 평가
(전반적인 수준을 2~3문장으로 요약)

### 📊 항목별 평가
| 항목 | 점수(1~5) | 피드백 |
|------|-----------|--------|
| 발음/억양 | X/5 | ... |
| 내용 완성도 | X/5 | ... |
| 어휘 | X/5 | ... |
| 문법 | X/5 | ... |
| 유창성 | X/5 | ... |

### ✅ 잘한 점
- (구체적으로 칭찬)

### ⚠️ 개선할 점
- (구체적인 개선 사항 및 수정 예시 제공)

### 💡 추천 표현
(더 자연스럽거나 고급스러운 대체 표현 2~3개 제안)

### 🏆 예상 등급
{exam_type} 기준 예상 등급: (등급 + 간단한 이유)
"""
    message_list = [{"role": "system", "content": system_prompt}]

    # 이전 대화 히스토리 반영
    # Gradio 6.9: {"role": "user"/"assistant", "content": ...} 딕셔너리 리스트 형식
    for msg in chat_history:
        role    = msg.get("role", "")
        content = msg.get("content", "")
        if role in ("user", "assistant") and content and content.strip():
            message_list.append({"role": role, "content": content})

    message_list.append({
        "role": "user",
        "content": f"다음 답변을 평가해주세요:\n\n\"{transcript}\"",
    })
    return call_openai(message_list)


# ============================================================
# Gradio UI
# ============================================================
with gr.Blocks(title="토익스피킹 / 오픽 AI 평가 챗봇") as demo:

    gr.Markdown("# 🎤 토익스피킹 / 오픽 AI 평가 챗봇")

    # gr.State: UI에 표시되지 않는 세션 변수
    # 현재 출제된 문제와 선택된 시험 유형을 저장해두고
    # 평가 함수에서 꺼내 쓴다
    current_question_state = gr.State("")
    exam_type_state        = gr.State("")

    # 챗봇
    chatbot = gr.Chatbot(label="AI 평가 챗봇", height=500)

    # 시험 / 파트 선택
    with gr.Row():
        exam_selector = gr.Radio(
            choices=["TOEIC Speaking", "OPIc"],
            label="📋 시험 유형",
            interactive=True,
        )
        part_selector = gr.Radio(
            choices=[],
            label="📂 파트 선택",
            interactive=False,
        )

    # 녹음 영역 (파트 선택 전에는 숨겨둠)
    with gr.Row(visible=False) as recording_row:
        prompt_audio = gr.Audio(
            label="🎤 녹음 (중지하면 자동 평가)",
            sources="microphone",
            type="filepath",
            scale=1,
        )
        transcript_box = gr.Textbox(
            label="📄 인식된 텍스트",
            placeholder="녹음을 중지하면 자동으로 표시됩니다.",
            interactive=False,
            scale=3,
        )

    clear_btn = gr.Button("🗑️ 처음부터 다시", variant="secondary")

    # ── 이벤트 핸들러 ─────────────────────────────────────────

    def on_load():
        """
        앱 최초 로드 시 호출.
        Gradio 구버전은 None을 허용하지 않으므로
        봇만 말하는 턴은 user 자리를 빈 문자열 ""로 채운다.
        → [["", bot_msg]] 형식
        """
        welcome = (
            "안녕하세요! 😊 공부하실 시험을 선택해주세요.\n"
            "**TOEIC Speaking** 또는 **OPIc** 중에서 골라주세요!"
        )
        return [{"role": "assistant", "content": welcome}]

    def on_exam_selected(exam_type, chat_history):
        """
        시험 유형 선택 시:
        1) part_selector의 choices를 해당 시험 파트 목록으로 교체
        2) 챗봇에 파트 선택 안내 메시지 추가
        3) exam_type_state에 선택값 저장
        """
        parts   = PART_CHOICES[exam_type]
        bot_msg = f"**{exam_type}**을 선택하셨네요! 👍\n이제 연습할 파트를 선택해주세요."
        return (
            gr.update(choices=parts, value=None, interactive=True),
            chat_history + [{"role": "assistant", "content": bot_msg}],
            exam_type,
        )

    def on_part_selected(part, exam_type, chat_history):
        """
        파트 선택 시:
        1) 해당 파트에서 예상 문제를 랜덤 1개 출제
        2) 챗봇에 문제 + 제한시간 메시지 추가
        3) 녹음 영역(recording_row) 표시
        4) current_question_state에 문제 저장
        """
        if not part:
            return chat_history, "", gr.update(visible=False)

        question, time_str = generate_question(part)

        bot_msg = (
            f"**{part}** 예상 문제를 드릴게요! 연습해보세요 :)\n\n"
            f"**문제)** {question}\n\n"
            f"**제한시간)** {time_str}"
        )
        return (
            chat_history + [{"role": "assistant", "content": bot_msg}],
            question,
            gr.update(visible=True),
        )

    def on_stop_recording(audio_path):
        """
        녹음 중지 시 Azure STT를 호출해 텍스트로 변환.
        결과를 transcript_box에 채운다.
        """
        if audio_path is None:
            return ""
        text = request_stt(audio_path)
        return text if text else "(음성 인식 실패 - 다시 시도해주세요)"

    def on_transcript_change(transcript, exam_type, current_question, chat_history):
        """
        transcript_box 값이 변경될 때 자동 실행.
        GPT에 평가를 요청하고 챗봇에 결과를 추가한다.
        - 빈 값이나 실패 메시지는 무시
        - 문제가 없는 경우(파트 미선택) 안내 메시지 출력
        """
        if not transcript or transcript.startswith("(음성 인식 실패"):
            return chat_history
        if not current_question:
            return chat_history + [{"role": "assistant", "content": "⚠️ 먼저 시험 유형과 파트를 선택해주세요!"}]

        bot_response = request_openai_evaluation(
            transcript, exam_type, current_question, chat_history
        )
        if not bot_response:
            return chat_history

        return chat_history + [
            {"role": "user",      "content": f"🎤 {transcript}"},
            {"role": "assistant", "content": bot_response},
        ]

    def on_clear():
        """
        처음부터 다시 버튼 클릭 시:
        모든 상태와 UI를 초기값으로 리셋한다.
        """
        welcome = (
            "안녕하세요! 😊 공부하실 시험을 선택해주세요.\n"
            "**TOEIC Speaking** 또는 **OPIc** 중에서 골라주세요!"
        )
        return (
            [{"role": "assistant", "content": welcome}],
            "",
            "",
            "",
            gr.update(value=None),
            gr.update(choices=[], value=None, interactive=False),
            gr.update(visible=False),
        )

    # ── 이벤트 연결 ───────────────────────────────────────────

    # 앱 로드 → 챗봇 첫 인사
    demo.load(fn=on_load, outputs=[chatbot])

    # 시험 선택 → 파트 목록 업데이트 + 안내 메시지
    exam_selector.change(
        fn=on_exam_selected,
        inputs=[exam_selector, chatbot],
        outputs=[part_selector, chatbot, exam_type_state],
    )

    # 파트 선택 → 문제 출제 + 녹음 영역 표시
    part_selector.change(
        fn=on_part_selected,
        inputs=[part_selector, exam_type_state, chatbot],
        outputs=[chatbot, current_question_state, recording_row],
    )

    # 녹음 중지 → STT → transcript_box
    prompt_audio.stop_recording(
        fn=on_stop_recording,
        inputs=[prompt_audio],
        outputs=[transcript_box],
    )

    # transcript_box 변경 → GPT 평가 → 챗봇
    transcript_box.change(
        fn=on_transcript_change,
        inputs=[transcript_box, exam_type_state, current_question_state, chatbot],
        outputs=[chatbot],
    )

    # 초기화 버튼
    clear_btn.click(
        fn=on_clear,
        outputs=[
            chatbot,
            transcript_box,
            current_question_state,
            exam_type_state,
            exam_selector,
            part_selector,
            recording_row,
        ],
    )

demo.launch()

* Running on local URL:  http://127.0.0.1:7932
* To create a public link, set `share=True` in `launch()`.


In [2]:
gr.__version__

'6.9.0'